# CrewAI Observability [Production - Module 02]

> **MLCourse - Agentic AI - CrewAI Production**

Observability is critical for production agentic systems. CrewAI provides
built-in tracing, event listeners, and OpenTelemetry integration to monitor
agent execution, debug failures, and measure performance. This notebook
covers all observability mechanisms and builds a custom event logger that
tracks agent execution times.

## What you will learn

1. CrewAI's built-in tracing system and how to enable it.
2. The `@event_listener` decorator and `crew.on_event()` callback system.
3. All event types: crew_start, crew_end, agent_start, agent_end, task_start, task_end.
4. Building a custom event logger that tracks agent execution times.
5. OpenTelemetry integration pattern for production monitoring.

## Key takeaways

- CrewAI tracing is enabled via environment variable `CREWAI_TRACING=true`.
- Event listeners let you hook into every stage of crew execution.
- Custom loggers can track wall-clock time per agent and per task.
- OpenTelemetry integration sends spans to Jaeger, Zipkin, or Grafana.

In [ ]:
# ---- Setup: imports, environment, track discovery ---------------------------

import os
import sys
import json
import time
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
load_dotenv(TRACK / ".env", override=False)

api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    print("[GREEN] OPENAI_API_KEY found -- optional cloud calls will work")
else:
    print("[GREEN] No API key needed -- using local ChatOllama")

In [ ]:
# ---- Check Ollama availability --------------------------------------------

OLLAMA_OK = False
try:
    from langchain_ollama import ChatOllama
    _test = ChatOllama(model="llama3.1:8b", temperature=0)
    _test.invoke("ping")
    OLLAMA_OK = True
    print("Ollama: ONLINE (llama3.1:8b)")
except Exception as e:
    print("Ollama: OFFLINE --", e)

In [ ]:
# ---- CrewAI imports ---------------------------------------------------------

try:
    from crewai import Agent, Task, Crew, Process, LLM
    from crewai.utilities.events import (
        CrewStartEvent,
        CrewEndEvent,
        AgentStartEvent,
        AgentEndEvent,
        TaskStartEvent,
        TaskEndEvent,
    )
    from crewai.utilities.events.base_event_listener import BaseEventListener
    CREWAI_OK = True
    print("CrewAI version:", __import__("crewai").__version__)
except ImportError as e:
    CREWAI_OK = False
    print("CrewAI not installed:", e)
except ImportError as e:
    # Some versions may not export all event classes.
    CREWAI_OK = True
    print("CrewAI loaded (some event classes may differ):", e)

## 1. Built-in Tracing

CrewAI has a built-in tracing system that records agent and task execution
details. Enable it by setting an environment variable before running your crew.

```bash
# Enable tracing via environment variable
export CREWAI_TRACING=true
```

Or set it in Python:
```python
os.environ["CREWAI_TRACING"] = "true"
```

Tracing output includes:
- Agent start/end timestamps and duration.
- Task start/end timestamps and duration.
- LLM calls and token usage.
- Tool invocations and results.
- Crew-level execution summary.

Traces are stored locally in a `.crewai/` directory by default.

In [ ]:
# ---- Enable built-in tracing -----------------------------------------------

# Set the tracing environment variable.
# This must be set BEFORE creating the crew.
os.environ["CREWAI_TRACING"] = "true"
print("CrewAI tracing enabled via CREWAI_TRACING=true")
print("Traces will be stored in .crewai/ directory")
print()

# Show where traces would be stored.
crewai_dir = Path.cwd() / ".crewai"
print(f"Trace storage directory: {crewai_dir}")
print("Note: Directory is created on first crew execution")

## 2. Event Types Reference

CrewAI emits events at each stage of execution. These events carry
metadata about the current agent, task, or crew.

| Event | When Emitted | Key Fields |
|-------|-------------|------------|
| `CrewStartEvent` | Crew begins execution | crew_name, inputs |
| `CrewEndEvent` | Crew finishes execution | crew_name, output, duration |
| `AgentStartEvent` | Agent begins a task | agent_role, task_description |
| `AgentEndEvent` | Agent finishes a task | agent_role, output |
| `TaskStartEvent` | Task begins execution | task_description, agent_role |
| `TaskEndEvent` | Task finishes execution | task_output, duration |

You can listen to these events using `crew.on_event()` or by subclassing
`BaseEventListener`.

In [ ]:
# ---- List all available event types ----------------------------------------

if CREWAI_OK:
    print("=== CrewAI Event Types ===\n")
    event_classes = [
        ("CrewStartEvent", "Crew begins execution"),
        ("CrewEndEvent", "Crew finishes execution"),
        ("AgentStartEvent", "Agent begins a task"),
        ("AgentEndEvent", "Agent finishes a task"),
        ("TaskStartEvent", "Task begins execution"),
        ("TaskEndEvent", "Task finishes execution"),
    ]
    for name, desc in event_classes:
        print(f"  {name:25s} -- {desc}")
    print()
    print("Events are subclasses of Pydantic BaseModel with typed fields.")

## 3. The `@event_listener` Decorator

The simplest way to listen for events is the `crew.on_event()` method.
You pass a callback function that receives the event object.

```python
def my_handler(event):
    print(f"Event: {type(event).__name__}")

crew.on_event("crew_start", my_handler)
crew.on_event("crew_end", my_handler)
```

Alternatively, subclass `BaseEventListener` and implement handler methods
for each event type. This is more structured and suitable for production.

In [ ]:
# ---- Demonstrate on_event callback pattern ---------------------------------

if CREWAI_OK and OLLAMA_OK:
    # Create a minimal crew for event demonstration.
    ollama_llm = LLM(model="ollama/llama3.1:8b", base_url="http://localhost:11434", temperature=0.7)

    monitor_agent = Agent(
        role="Monitor",
        goal="Perform a simple verification task.",
        backstory="You are a monitoring agent that checks system status.",
        llm=ollama_llm,
        verbose=False,
        allow_delegation=False,
    )

    monitor_task = Task(
        description="Verify that the monitoring system is operational. Report status.",
        expected_output="A status report indicating the system is operational.",
        agent=monitor_agent,
    )

    monitor_crew = Crew(
        agents=[monitor_agent],
        tasks=[monitor_task],
        process=Process.sequential,
        verbose=False,
    )

    # Event tracking list -- callbacks will append to this.
    events_log = []

    def on_crew_start(event):
        events_log.append({"type": "crew_start", "time": time.time(), "data": str(event)})
        print("[EVENT] Crew started")

    def on_crew_end(event):
        events_log.append({"type": "crew_end", "time": time.time(), "data": str(event)})
        print("[EVENT] Crew ended")

    def on_agent_start(event):
        events_log.append({"type": "agent_start", "time": time.time(), "data": str(event)})
        print("[EVENT] Agent started:", getattr(event, "agent_role", "unknown"))

    def on_agent_end(event):
        events_log.append({"type": "agent_end", "time": time.time(), "data": str(event)})
        print("[EVENT] Agent ended:", getattr(event, "agent_role", "unknown"))

    # Register event handlers.
    monitor_crew.on_event("crew_start", on_crew_start)
    monitor_crew.on_event("crew_end", on_crew_end)
    monitor_crew.on_event("agent_start", on_agent_start)
    monitor_crew.on_event("agent_end", on_agent_end)

    print("Event handlers registered. Running crew...")
    print("=" * 60)
    result = monitor_crew.kickoff()
    print("=" * 60)
    print(f"\nTotal events captured: {len(events_log)}")
    for evt in events_log:
        print(f"  {evt['type']}: {evt['time']:.3f}")
else:
    print("[SKIP] CrewAI or Ollama not available")

## 4. Custom Event Logger with Execution Time Tracking

For production, you need a structured logger that captures timing data
for every agent and task. We build a `TimingLogger` that subclasses
`BaseEventListener` and records wall-clock durations.

In [ ]:
# ---- Build a custom TimingLogger class -------------------------------------

if CREWAI_OK:
    class TimingLogger(BaseEventListener):
        """Custom event listener that tracks agent and task execution times.

        Records wall-clock duration for each agent and task, and maintains
        a running log of all events for later analysis.
        """

        setup_events = [
            "crew_start", "crew_end",
            "agent_start", "agent_end",
            "task_start", "task_end",
        ]

        def __init__(self):
            super().__init__()
            self.timeline = []          # ordered list of all events
            self.agent_starts = {}      # role -> start_time
            self.agent_durations = {}   # role -> [durations]
            self.task_starts = {}       # desc -> start_time
            self.task_durations = {}    # desc -> [durations]
            self.crew_start_time = None
            self.crew_end_time = None

        def _record(self, event_type, **kwargs):
            entry = {"event": event_type, "timestamp": time.time(), **kwargs}
            self.timeline.append(entry)
            return entry

        def crew_start(self, event):
            self.crew_start_time = time.time()
            self._record("crew_start")
            print(f"  [TimingLogger] Crew started at {self.crew_start_time:.3f}")

        def crew_end(self, event):
            self.crew_end_time = time.time()
            duration = self.crew_end_time - (self.crew_start_time or self.crew_end_time)
            self._record("crew_end", duration_s=round(duration, 3))
            print(f"  [TimingLogger] Crew ended -- total {duration:.3f}s")

        def agent_start(self, event):
            role = getattr(event, "agent_role", "unknown")
            self.agent_starts[role] = time.time()
            self._record("agent_start", agent=role)
            print(f"  [TimingLogger] Agent '{role}' started")

        def agent_end(self, event):
            role = getattr(event, "agent_role", "unknown")
            start = self.agent_starts.pop(role, time.time())
            duration = time.time() - start
            self.agent_durations.setdefault(role, []).append(duration)
            self._record("agent_end", agent=role, duration_s=round(duration, 3))
            print(f"  [TimingLogger] Agent '{role}' ended -- {duration:.3f}s")

        def task_start(self, event):
            desc = getattr(event, "task_description", "unknown")[:50]
            self.task_starts[desc] = time.time()
            self._record("task_start", task=desc)
            print(f"  [TimingLogger] Task '{desc}' started")

        def task_end(self, event):
            desc = getattr(event, "task_description", "unknown")[:50]
            start = self.task_starts.pop(desc, time.time())
            duration = time.time() - start
            self.task_durations.setdefault(desc, []).append(duration)
            self._record("task_end", task=desc, duration_s=round(duration, 3))
            print(f"  [TimingLogger] Task '{desc}' ended -- {duration:.3f}s")

        def summary(self):
            """Print a summary of all recorded timing data."""
            print("\n=== TimingLogger Summary ===")
            if self.crew_start_time and self.crew_end_time:
                total = self.crew_end_time - self.crew_start_time
                print(f"Total crew time: {total:.3f}s")
            print(f"Events recorded: {len(self.timeline)}")
            print(f"Agents tracked: {len(self.agent_durations)}")
            for role, durations in self.agent_durations.items():
                avg = sum(durations) / len(durations)
                print(f"  {role}: avg={avg:.3f}s ({len(durations)} runs)")
            print(f"Tasks tracked: {len(self.task_durations)}")
            for desc, durations in self.task_durations.items():
                avg = sum(durations) / len(durations)
                print(f"  {desc}: avg={avg:.3f}s ({len(durations)} runs)")

        def to_json(self):
            """Export timeline as JSON for external analysis."""
            return json.dumps(self.timeline, indent=2, default=str)

    print("TimingLogger class defined")
    print("Methods: crew_start, crew_end, agent_start, agent_end,")
    print("         task_start, task_end, summary, to_json")

## 5. Using the TimingLogger with a Crew

We wire the `TimingLogger` into a crew and run it. The logger prints
events as they happen and provides a summary at the end.

In [ ]:
# ---- Run a crew with the TimingLogger attached -----------------------------

if CREWAI_OK and OLLAMA_OK:
    ollama_llm = LLM(model="ollama/llama3.1:8b", base_url="http://localhost:11434", temperature=0.7)

    timed_agent = Agent(
        role="Analyst",
        goal="Analyze the given data point and provide insights.",
        backstory="You are a data analyst who provides clear, concise insights.",
        llm=ollama_llm,
        verbose=False,
        allow_delegation=False,
    )

    timed_task = Task(
        description="Analyze the trend: AI adoption increased 40% in 2025.",
        expected_output="A 2-3 sentence analysis of the AI adoption trend.",
        agent=timed_agent,
    )

    timed_crew = Crew(
        agents=[timed_agent],
        tasks=[timed_task],
        process=Process.sequential,
        verbose=False,
    )

    # Create and attach the timing logger.
    logger = TimingLogger()
    timed_crew.add_event_listener(logger)

    print("Running crew with TimingLogger attached...")
    print("=" * 60)
    result = timed_crew.kickoff()
    print("=" * 60)

    # Print the summary.
    logger.summary()
    TIMING_LOGGER_OK = True
else:
    print("[SKIP] CrewAI or Ollama not available")
    TIMING_LOGGER_OK = False

## 6. Exporting Timing Data to JSON

For production dashboards, you need structured data. The `TimingLogger`
exports its timeline as JSON, which can be ingested by Grafana, Datadog,
or custom dashboards.

In [ ]:
# ---- Export timing data to JSON -------------------------------------------

if CREWAI_OK and TIMING_LOGGER_OK:
    # Export the timeline to a JSON file.
    data_dir = TRACK / "03_agentic_ai" / "04_crewai" / "data"
    data_dir.mkdir(parents=True, exist_ok=True)
    trace_file = data_dir / "crew_trace.json"

    with open(trace_file, "w", encoding="utf-8") as f:
        f.write(logger.to_json())

    print(f"Trace data exported to: {trace_file}")
    print(f"Events exported: {len(logger.timeline)}")

    # Show the first few events.
    print("\nFirst 3 events:")
    for entry in logger.timeline[:3]:
        print(json.dumps(entry, indent=2, default=str)[:200])
        print()
else:
    print("[SKIP] TimingLogger not available")

## 7. OpenTelemetry Integration Pattern

For production systems, OpenTelemetry (OTel) provides standardized
distributed tracing. CrewAI can integrate with OTel to send spans
to backends like Jaeger, Zipkin, or Grafana Tempo.

The pattern:
1. Install `opentelemetry-api` and a tracer provider.
2. Create a span for each crew, agent, and task execution.
3. Set span attributes (agent role, task description, etc.).
4. Export spans to your tracing backend.

In [ ]:
# ---- OpenTelemetry integration pattern (reference code) --------------------

print("=== OpenTelemetry Integration Pattern ===\n")
print("Reference implementation (requires opentelemetry-api):\n")

otel_code = '''
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter

# 1. Set up tracer provider.
provider = TracerProvider()
exporter = OTLPSpanExporter(endpoint="localhost:4317")
provider.add_span_processor(BatchSpanProcessor(exporter))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("crewai-production")

# 2. Use in event listeners.
class OTelListener(BaseEventListener):
    setup_events = ["crew_start", "crew_end", "agent_start", "agent_end"]

    def crew_start(self, event):
        self._span = tracer.start_span("crew_execution")

    def crew_end(self, event):
        self._span.end()

    def agent_start(self, event):
        role = getattr(event, "agent_role", "unknown")
        self._agent_span = tracer.start_span(
            f"agent_{role}",
            parent=self._span,
        )

    def agent_end(self, event):
        self._agent_span.end()
'''
print(otel_code)
print("\nThis pattern sends spans to Jaeger/Zipkin/Grafana Tempo.")

## 8. Custom Logging with Python's `logging` Module

For simpler setups, you can use Python's built-in logging module alongside
CrewAI's event system. This gives you structured logs without external
infrastructure.

In [ ]:
# ---- Custom logging integration pattern ------------------------------------

import logging

# Configure structured logging.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger_std = logging.getLogger("crewai_monitor")

if CREWAI_OK:
    class LoggingEventListener(BaseEventListener):
        """Logs all crew events using Python's logging module."""

        setup_events = [
            "crew_start", "crew_end",
            "agent_start", "agent_end",
            "task_start", "task_end",
        ]

        def crew_start(self, event):
            logger_std.info("CREW_START: execution began")

        def crew_end(self, event):
            logger_std.info("CREW_END: execution finished")

        def agent_start(self, event):
            role = getattr(event, "agent_role", "unknown")
            logger_std.info(f"AGENT_START: role={role}")

        def agent_end(self, event):
            role = getattr(event, "agent_role", "unknown")
            logger_std.info(f"AGENT_END: role={role}")

        def task_start(self, event):
            desc = getattr(event, "task_description", "unknown")[:60]
            logger_std.info(f"TASK_START: {desc}")

        def task_end(self, event):
            desc = getattr(event, "task_description", "unknown")[:60]
            logger_std.info(f"TASK_END: {desc}")

    print("LoggingEventListener defined")
    print("Uses Python logging module -- no external dependencies needed")
    print()
    print("Sample log output:")
    logger_std.info("CREW_START: execution began")
    logger_std.info("AGENT_START: role=Analyst")
    logger_std.info("TASK_START: Analyze the trend...")
    logger_std.info("TASK_END: Analyze the trend...")
    logger_std.info("AGENT_END: role=Analyst")
    logger_std.info("CREW_END: execution finished")
else:
    print("[SKIP] CrewAI not available")

## 9. Combining Multiple Listeners

In production, you often need multiple listeners running simultaneously:
one for timing, one for logging, and one for OpenTelemetry. CrewAI
supports attaching multiple listeners to a single crew.

In [ ]:
# ---- Demonstrate multiple listener attachment ------------------------------

if CREWAI_OK:
    print("=== Multiple Listener Pattern ===\n")
    print("Attach multiple listeners to one crew:\n")
    print("  timing_logger = TimingLogger()")
    print("  logging_listener = LoggingEventListener()")
    print("  otel_listener = OTelListener()")
    print()
    print("  crew.add_event_listener(timing_logger)")
    print("  crew.add_event_listener(logging_listener)")
    print("  crew.add_event_listener(otel_listener)")
    print()
    print("All listeners receive every event independently.")
    print("Order of execution: attachment order (FIFO).")

## 10. Production Observability Checklist

Before deploying a CrewAI system to production, ensure you have:

- [ ] Tracing enabled (`CREWAI_TRACING=true`).
- [ ] Custom event listener for timing metrics.
- [ ] Structured logging via Python `logging` or OpenTelemetry.
- [ ] Error handling in event listeners (wrap in try/except).
- [ ] Metric export to your monitoring system (Prometheus, Datadog, etc.).
- [ ] Alert thresholds for agent execution time.
- [ ] Dashboard showing crew success/failure rates.
- [ ] Log rotation and retention policy.

In [ ]:
# ---- Production checklist summary -----------------------------------------

print("=== Production Observability Checklist ===\n")
checklist = [
    ("Tracing enabled", "CREWAI_TRACING=true"),
    ("Custom timing listener", "TimingLogger class"),
    ("Structured logging", "Python logging or OTel"),
    ("Error handling", "try/except in listeners"),
    ("Metric export", "Prometheus/Datadog/Grafana"),
    ("Alert thresholds", "Agent time > 30s triggers alert"),
    ("Dashboard", "Crew success rate, latency P50/P95/P99"),
    ("Log retention", "30-day retention, rotate daily"),
]
for item, detail in checklist:
    print(f"  [x] {item:25s} -- {detail}")

## Summary

This notebook covered all CrewAI observability mechanisms:

1. **Built-in tracing** -- `CREWAI_TRACING=true` for automatic trace recording.
2. **Event types** -- crew_start/end, agent_start/end, task_start/end.
3. **Event listeners** -- `crew.on_event()` and `BaseEventListener` subclasses.
4. **TimingLogger** -- custom listener tracking agent and task durations.
5. **JSON export** -- structured trace data for dashboards.
6. **OpenTelemetry** -- standard distributed tracing with Jaeger/Zipkin.
7. **Python logging** -- lightweight structured logs without infrastructure.
8. **Multiple listeners** -- combine timing, logging, and OTel simultaneously.

## Next steps

- Integrate with a real tracing backend (Jaeger, Grafana Tempo).
- Add Prometheus metrics export for alerting.
- Build a real-time dashboard with Streamlit showing crew execution times.